# Protocol check: does the 10.8 pp FFT–LoRA gap depend on scoring and option order?

Qwen2.5-0.5B, main order, Step 0 and Step 3, seeds 42/43/44. Runtime: **T4 GPU** (same as Assignment 4). Every cell is resumable: if Colab disconnects, run all cells again and finished checkpoints are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/rtm_a4'
import os; os.makedirs(BASE, exist_ok=True)

In [ ]:
%cd /content
!rm -rf Catastrophic-Forgetting-Fft-vs-Lora
!git clone -q https://github.com/astmirzhan/Catastrophic-Forgetting-Fft-vs-Lora
%cd Catastrophic-Forgetting-Fft-vs-Lora
!git log -1 --oneline
!ls protocol

Same library versions as the Assignment 4 Colab runs (`results_colab/pip_freeze.txt`). After this cell: *Runtime → Restart session*, then continue from the next cell (re-run the `%cd` line).

In [ ]:
!pip install -q transformers==5.21.0 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0 tabulate

## Day 1 — checkpoint availability and Step 0 reproduction

Assignment 4 did not save model weights (`save_strategy="no"`), so Step 3 must be re-trained. Step 0 is the base model. This cell scores Step 0 in all three modes and compares the original letter score item by item with `results_colab/baselines/qwen0.5b.json`. Expected: `diff_pp 0.0`, `same_prediction_pct 100.0` (≈ 51.9%).

In [ ]:
%cd /content/Catastrophic-Forgetting-Fft-vs-Lora
OUT = f'{BASE}/results_protocol'
!python protocol/protocol_eval.py --step0-only --out {OUT}
!cat {OUT}/repro/qwen0.5b_step0.json

## Days 1–4 — re-train Step 3 and run the three evaluations

≈ 12 min training + ≈ 10 min evaluation per checkpoint, 6 checkpoints ≈ 2–2.5 h. Weights are saved to Drive (≈ 2 GB each, fp32) so the checkpoints exist for the final project. Drop `--save-weights` if Drive is full. If time runs out, run `--seeds 42` only and report reduced coverage.

In [ ]:
WEIGHTS = f'{BASE}/ckpt_step3'
!python protocol/protocol_eval.py --seeds 42 43 44 --out {OUT} --save-weights {WEIGHTS} 2>&1 | grep -v 'it/s'

## Day 5 — table, figure, reproduction summary

In [ ]:
REP = f'{BASE}/report_protocol'
!python protocol/protocol_analysis.py --res {OUT} --out {REP}
from IPython.display import Image, Markdown, display
display(Image(f'{REP}/fig_gap_by_mode.png'))
display(Markdown(open(f'{REP}/tables.md').read()))

## Joint check of one table row against the item-level CSV

Recounts correct/total directly from the CSV rows; compare with the same row in `table_protocol.csv`.

In [ ]:
!python protocol/protocol_analysis.py --res {OUT} --verify letter_shuf FFT 42 All
!grep '^letter_shuf,All,42,FFT' {REP}/table_protocol.csv